# Prerequisites

In [ ]:
# Download data
!wget https://s3.amazonaws.com/opennmt-trainingdata/toy-ende.tar.gz
!tar xf toy-ende.tar.gz

# Sentencepiece

In [ ]:
!pip install sentencepiece

In [ ]:
# Train
import sentencepiece as spm

spm.SentencePieceTrainer.train(
  input='toy-ende/src-train.txt',
  model_prefix='english_sp_model',
  vocab_size=4000,
)

spm.SentencePieceTrainer.train(
  input='toy-ende/tgt-train.txt',
  model_prefix='german_sp_model',
  vocab_size=4000,
)

This creates 4 files:

* english_sp_model.model
* english_sp_model.vocab
* german_sp_model.model
* german_sp_model.vocab


# OpenNMT + Sentencepiece

## Step 0: Install OpenNMT

In [ ]:
!pip install OpenNMT-py
!pip install "numpy<2.0" # This fixes an error caused by OpenNMT not being maintained

## Step 1: Prepare the data

In [ ]:
# Build YAML Config file
yaml_text = """
# toy_en_de.yaml

## Where the samples will be written
save_data: toy-ende/run/example

src_vocab: toy-ende/run/example.vocab.src
tgt_vocab: toy-ende/run/example.vocab.tgt

# Prevent overwriting existing files in the folder
overwrite: True

# -- Sentencepiece Params --
# Tokenization options
src_subword_type: sentencepiece
src_subword_model: english_sp_model.model
tgt_subword_type: sentencepiece
tgt_subword_model: german_sp_model.model

# Number of candidates for SentencePiece sampling
subword_nbest: 20
# Smoothing parameter for SentencePiece sampling
subword_alpha: 0.1
# Specific arguments for pyonmttok
src_onmttok_kwargs: "{'mode': 'none', 'spacer_annotate': True}"
tgt_onmttok_kwargs: "{'mode': 'none', 'spacer_annotate': True}"


# Corpus opts:
data:
    corpus_1:
        path_src: toy-ende/src-train.txt
        path_tgt: toy-ende/tgt-train.txt
        transforms: [onmt_tokenize]
    valid:
        path_src: toy-ende/src-val.txt
        path_tgt: toy-ende/tgt-val.txt
        transforms: [onmt_tokenize]

# Train on a single GPU
bucket_size: 10000
world_size: 1
gpu_ranks: [0]

# Where to save the checkpoints
save_model: toy-ende/run/model
save_checkpoint_steps: 500
train_steps: 1000
valid_steps: 500
"""

with open("toy_en_de.yaml", "w") as f:
    f.write(yaml_text)

In [ ]:
!onmt_build_vocab -config toy_en_de.yaml -n_sample -1

## Step 2: Train the model

In [ ]:
!onmt_train -config toy_en_de.yaml

## Step 3: Translate

This time we'll be using the sentencepiece parameters in our onmt_translate command

In [ ]:
translate_yaml_text = """
model: toy-ende/run/model_step_1000.pt
src: toy-ende/src-test.txt
output: toy-ende/pred_1000.txt
gpu: 0
verbose: True

# Sentencepiece Params

# Tokenization options
transforms: onmt_tokenize
src_subword_type: sentencepiece
src_subword_model: english_sp_model.model
tgt_subword_type: sentencepiece
tgt_subword_model: german_sp_model.model

src_subword_nbest: 20
src_subword_alpha: 0.1
"""

with open("translate_yaml.yaml", "w") as f:
    f.write(translate_yaml_text)

In [ ]:
!onmt_translate -config translate_yaml.yaml

Check your `toy-ende/pred_1000.txt` file for the detokenized output of the model.